# 🏥 KUH Ameliyat Vaka Hacmi Tahmini — v5 (2020–2026)
### LSTM · Facebook Prophet · LightGBM · Weighted Ensemble
**Kaynak:** Bui et al. (2023) — *IISE Transactions on Healthcare Systems Engineering*

---

## 📊 Veri Seti Profili
| Özellik | Değer |
|---|---|
| **Ameliyat Verisi** | KUH_2020_2026_Ameliyat.xlsx (68,473 kayıt) |
| **Doluluk Verisi** | KUH_Hastane_Doluluk_Oranı.xlsx (Ara 2022 – Kas 2025, günlük) |
| **Eğitim** | Oca 2020 – Ara 2024 · **Validasyon** | Oca 2025 – Ara 2025 |
| **Tahmin** | Oca 2026 – Mar 2026 |

## ✅ v4'e göre değişiklikler (Kapsamlı Tuning — v5)
| # | Alan | Değişiklik | Beklenen etki |
|---|---|---|---|
| 1 | **LSTM** | `learning_rate 1e-3→3e-4`, `clipnorm 1.0→5.0` | Daha stabil convergence |
| 2 | **LSTM** | `recurrent_dropout` parametreye çıkarıldı (0.1) | Ayrı tuning imkânı |
| 3 | **LSTM** | Huber `delta=1.0` parametreye çıkarıldı | Aykırı değer hassasiyeti |
| 4 | **LSTM** | Decoder'a 2. LSTM katmanı eklendi | Daha derin kod çözme |
| 5 | **Scaler** | `MinMaxScaler → RobustScaler` | Pandemi dönemi aykırılıklarına dayanıklı |
| 6 | **GBM** | `sklearn GBM → LightGBM` | 10× hız, daha iyi performans |
| 7 | **Özellik** | Tatil penceresi `±2→±3`, lag `{3,60,90}` eklendi | Daha uzun bağlam |
| 8 | **Özellik** | `EWMA(span=7)` doluluk özelliği eklendi | Son günlere ağırlık |
| 9 | **Özellik** | COVID sınırı parametreye çıkarıldı | Kolayca test edilebilir |
| 10 | **Prophet** | `seasonality_mode→multiplicative`, `yearly_seasonality→15` | Mevsimsel ölçek etkisi |
| 11 | **Ensemble** | Val R²'ye göre ağırlıklı birleştirme eklendi | En iyi modellerin etkisi artar |

---
### ▶️ Kullanım
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. Her iki Excel dosyasını sırayla yükle

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Bağımlılıkları Kur                                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   Google Colab ortamına varsayılan olarak yüklü gelmeyen üç paketi kurar:
#   • prophet  — Facebook'un zaman serisi tahmin kütüphanesi
#   • openpyxl — Excel (.xlsx) dosyalarını okumak için
#   • lightgbm — Hızlı ve bellek açısından verimli Gradient Boosting ağaçları
#
# Değiştirilecek bir şey yok. Lokal ortamda çalıştırıyorsan bu hücreyi
# atlayabilirsin (paketlerin zaten kuruluysa). Kurulum başarısız olursa
# 'Runtime → Restart and run all' ile tekrar dene.
#
!pip install prophet openpyxl lightgbm -q
print('✔ Kurulum tamamlandı (prophet · openpyxl · lightgbm)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Import & Merkezi Konfigürasyon Paneli                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   Tüm kütüphaneleri tek seferde içe aktarır ve projedeki tüm parametreleri
#   tek bir yerde toplar. Diğer hücreler bu değişkenleri doğrudan referans
#   alır — parametre denemelerinde sadece BU hücreyi düzenleyip 'Run All'
#   yapmak yeterlidir.
#
import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, RepeatVector,
    TimeDistributed, Bidirectional
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from prophet import Prophet
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import lightgbm as lgb

# ─────────────────────────────────────────────────────────────────────────────
# ⚙️  TUNING PANELİ — tüm parametreler tek yerde
# ─────────────────────────────────────────────────────────────────────────────

# ── LSTM — Mimari ─────────────────────────────────────────────────────────────
SEED              = 42

LOOKBACK          = 28
# [TEST ET] Geçmişe bakış penceresi (gün). Model bu kadar günü 'görerek' tahmin yapar.
# Dene: 21 → 28 → 42 → 56
# Öneri: 42 veya 56 dene — 2 aylık hafıza, mevsimsel örüntüler (Ramazan, yaz tatili)
# için 28 günden daha güçlü bağlam sağlayabilir. Artırdıkça eğitim süresi de artar.

HORIZON           = 7
# [SABİT] Kaç gün ileriye tahmin yapılacağı. 7 = haftalık blok.
# Değiştirirsen direct_forecast_lgb ve make_seq_flat fonksiyonları da güncellenmeli.

LSTM_UNITS        = 64
# [TEST ET] Encoder ve decoder LSTM katmanlarının genişliği (gizli birim sayısı).
# Dene: 64 → 96 → 128
# Öneri: Özellik sayısı yüksek olduğundan 96 veya 128 daha zengin temsil öğrenebilir.
# GPU kullanıyorsan maliyet düşük.

DROPOUT           = 0.3
# [TEST ET] İleri besleme katmanlarındaki dropout oranı (regularizasyon).
# Dene: 0.2 → 0.3 → 0.4
# Öneri: Val kaybı eğitim kaybından çok yüksekse (overfitting) 0.4'e çık.
# Az verili periyotlarda 0.2 daha iyi öğrenme sağlayabilir.

RECURRENT_DROPOUT = 0.1
# [TEST ET] LSTM gizli durum bağlantılarındaki dropout (zaman boyutu regularizasyonu).
# Dene: 0.0 → 0.1 → 0.2
# Öneri: GPU kullanıyorsan 0.0 genellikle daha hızlı ve benzer performans verir
# (cuDNN optimizasyonu recurrent_dropout>0 ile devre dışı kalır). Overfitting
# görülüyorsa 0.2'ye çık.

HUBER_DELTA       = 1.0
# [TEST ET] Huber kayıp fonksiyonunun eşik değeri (delta).
# Delta altındaki hatalar MSE gibi (pürüzsüz gradyan), üstündekiler MAE gibi (aykırı değere dayanıklı).
# Dene: 0.5 → 1.0 → 2.0
# Öneri: Pandemi döneminde çok büyük sıfır-vaka günleri varsa 0.5 daha sağlam;
# veri temizse 2.0 standart MSE'ye yaklaşır.

EPOCHS            = 200
# EarlyStopping aktif olduğundan bu değer maksimum; genellikle çok önce durur.

BATCH_SIZE        = 16
# [TEST ET] Her gradient adımında işlenen dizi sayısı.
# Dene: 16 → 32
# Öneri: 32 daha düzgün gradyan verir ve eğitimi hızlandırır.
# 16 daha stokastik ama bazen daha iyi genelleşir.

LEARNING_RATE     = 3e-4
# [OPTİMUM] v5'te 1e-3'ten düşürüldü. ReduceLROnPlateau dinamik olarak daha da
# düşüreceği için bu başlangıç değeri dengeli. Exploding gradient görülmüyorsa
# değiştirmene gerek yok.

CLIP_NORM         = 5.0
# [OPTİMUM] Gradient norm kırpma üst sınırı. v5'te 1.0'dan 5.0'a gevşetildi.
# Eğitim kaybında ani sıçrama (exploding gradient) görülürse 1.0'a düşür.

EARLY_PATIENCE    = 30
# [OPTİMUM] Val kaybı bu kadar epoch iyileşmezse eğitim durur.
# Dene: 25 → 30 → 40
# Huber kaybıyla eğitim dalgalı olabilir; 30 bu dengeyi iyi kuruyor.

REDUCE_FACTOR     = 0.5
# [TEST ET] ReduceLROnPlateau: plato tespitinde LR'yi bu katsayıyla çarpar.
# Dene: 0.3 → 0.5
# Öneri: 0.3 daha agresif LR düşürümü; kayıp eğrisinde platoya daha erken
# oturup oturmadığını izle.

REDUCE_PATIENCE   = 15
# [OPTİMUM] LR düşürme için plato toleransı (epoch sayısı).
# Dene: 10 → 15 → 20

VAL_SPLIT         = 0.15
# [TEST ET] LSTM iç validasyonu için eğitim setinden ayrılan oran.
# Dene: 0.10 → 0.15 → 0.20
# Öneri: Eğitim verisi az görünüyorsa 0.10'a düşür; overfitting izliyorsan 0.20'ye çık.

# ── Özellik mühendisliği ───────────────────────────────────────────────────────
COVID_END         = '2022-06-01'
# [TEST ET] Pandemi 'bitti' bayrağının sıfırlandığı tarih.
# Dene: '2022-03-01' → '2022-06-01' → '2022-09-01'
# Öneri: KUH'taki vaka hacmi pandemi normalleşmesini hangi tarihte gösteriyor?
# Kendi veri görselinden bu tarihi belirle.

HOL_WINDOW        = 3
# [OPTİMUM] Tatil gününden ±kaç günlük yakınlık bayrağı üretilsin.
# Dene: 2 → 3 → 4
# Türk tatillerinde 3 gün öncesi/sonrası yeterli; büyük bayramlarda 4 denenebilir.

DOL_EWMA_SPAN     = 7
# [TEST ET] Doluluk oranı için üstel ağırlıklı hareketli ortalama penceresi.
# Dene: 3 → 7 → 14 → 30
# Öneri: 3 son günlere çok hızlı tepki verir (gürültülü olabilir);
# 14 daha düzgün ama gecikmeli bir sinyal sağlar.

# ── LightGBM ──────────────────────────────────────────────────────────────────
LGB_N_ESTIMATORS  = 300
# [TEST ET] Toplam ağaç (iterasyon) sayısı.
# Dene: 200 → 300 → 500
# Öneri: Early stopping eklenirse (Cell 11'e bak) 500-1000'e çıkabilirsin.
# Early stopping yoksa 500 ile overfitting riski var.

LGB_MAX_DEPTH     = 6
# [TEST ET] Her ağacın maksimum derinliği.
# Dene: 4 → 6 → 8
# Öneri: 4 daha basit ve genelleştirilmiş; 8 overfitting riski taşır,
# yalnızca büyük veri setlerinde dene.

LGB_LR            = 0.05
# [OPTİMUM] LightGBM öğrenme hızı. N_ESTIMATORS ile birlikte ayarla:
# düşük LR + çok ağaç = daha iyi performans (örn. 0.03 + 500).

LGB_SUBSAMPLE     = 0.8
# [OPTİMUM] Her ağaç için örneklenen satır oranı (row subsampling).
# Dene: 0.6 → 0.8 → 0.9
# Pandemi döneminde seyrek/aykırı veri varsa 0.6 daha dayanıklı olabilir.

LGB_MIN_LEAF      = 5
# [TEST ET] Yaprak düğümünde bulunması gereken minimum örnek sayısı.
# Dene: 3 → 5 → 10
# Öneri: 10 daha agresif regularizasyon; hafta sonu/tatil gibi
# az örnekli durumlar için 3 daha iyi fit sağlayabilir.

LGB_COLSAMPLE     = 0.8
# [OPTİMUM] Her ağaç için örneklenen sütun (özellik) oranı.
# Dene: 0.6 → 0.8 → 1.0
# Özellik sayısı yüksek olduğundan 0.8 iyi; korelasyonlu özellikler
# varsa 0.6 daha çeşitli ağaçlar üretir.

# ── Prophet ───────────────────────────────────────────────────────────────────
PROPHET_CP_SCALE  = 0.15
# [TEST ET] EN KRİTİK PROPHET PARAMETRESİ — trend değişim noktalarının esnekliği.
# Dene: 0.05 → 0.15 → 0.3 → 0.5
# Öneri: 0.05 trend değişimlerine sığır kalır; 0.5 overfit eder.
# Pandemi sonrası yapısal kırılmalar için 0.3 denemeye değer.
# Bileşen grafiğinde trend düz/garip görünüyorsa bu değeri ayarla.

PROPHET_SEAS_MODE = 'multiplicative'
# [OPTİMUM] v5'te 'additive'den değiştirildi.
# Ameliyat hacmi yıl içinde oransal değişim gösteriyorsa multiplicative doğru seçim.

PROPHET_SEAS_PRIOR= 10.0
# [TEST ET] Mevsimsellik bileşenlerinin esneklik sınırı.
# Dene: 5.0 → 10.0 → 20.0
# Haftalık mevsimsellik (Cuma/hafta sonu düşüşü) güçlüyse 20.0 dene.

PROPHET_HOL_PRIOR = 15.0
# [OPTİMUM] Tatil etkisinin büyüklük sınırı.
# Türk tatillerinde vaka sıfıra yakın düşüyorsa 20.0 daha sert etki modeller.
# Prophet bileşen grafiğindeki tatil bileşenini kontrol et.

PROPHET_YEARLY_N  = 15
# [TEST ET] Yıllık mevsimsellik için Fourier bileşeni sayısı.
# Dene: 10 → 15 → 20
# 20 ince mevsimsellikleri yakalıyor ama overfit riski var;
# 5 yıllık veri için 15 makul, 10 ile kıyasla.

# ── Dönem sınırları ───────────────────────────────────────────────────────────
TRAIN_END      = '2025-01-01'
VAL_END        = '2026-01-01'
FORECAST_START = '2026-01-01'
FORECAST_END   = '2026-03-31'

print('✔ Konfigürasyon yüklendi')
print(f'  LOOKBACK={LOOKBACK}  LSTM_UNITS={LSTM_UNITS}  DROPOUT={DROPOUT}')
print(f'  LR={LEARNING_RATE}  clipnorm={CLIP_NORM}  Huber_δ={HUBER_DELTA}')
print(f'  LGB: trees={LGB_N_ESTIMATORS}  depth={LGB_MAX_DEPTH}  lr={LGB_LR}')
print(f'  Prophet: cp_scale={PROPHET_CP_SCALE}  mode={PROPHET_SEAS_MODE}  yearly_N={PROPHET_YEARLY_N}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Dosya Yükleme (Google Colab)                                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   Google Colab'ın dosya yükleme arayüzünü açar ve iki Excel dosyasını
#   sırayla belleğe alır:
#     1. KUH_2020_2026_Ameliyat.xlsx  — her satırı bir ameliyat kaydı (68.473 kayıt)
#     2. KUH_Hastane_Doluluk_Oranı.xlsx — Ara 2022-Kas 2025 günlük doluluk oranı
#
# Lokal ortama taşıma:
#   Bu hücreyi kaldırıp aşağıdaki iki satırı Cell 2'nin sonuna ekle:
#     DATA_PATH   = '/tam/yol/KUH_2020_2026_Ameliyat.xlsx'
#     DOLULUK_PATH = '/tam/yol/KUH_Hastane_Doluluk_Oranı.xlsx'
#
from google.colab import files

print('📂 Lütfen yükleyin: KUH_2020_2026_Ameliyat.xlsx')
uploaded1  = files.upload()
DATA_PATH  = list(uploaded1.keys())[0]
print(f'✔ Yüklendi: {DATA_PATH}')

print()
print('📂 Lütfen yükleyin: KUH_Hastane_Doluluk_Oranı.xlsx')
uploaded2    = files.upload()
DOLULUK_PATH = list(uploaded2.keys())[0]
print(f'✔ Yüklendi: {DOLULUK_PATH}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Veri Hazırlama & Doluluk Entegrasyonu                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar — üç aşama:
#
# 4a — Ameliyat verisi:
#   Excel okunur → C sütunu (iloc[:,2]) tarih olarak parse edilir →
#   groupby(date).size() ile her gün için toplam vaka sayısı (volume) üretilir →
#   eksik günler fill_value=0 ile tam takvim serisine dönüştürülür.
#
# 4b — Doluluk verisi:
#   İki sütunlu Excel (tarih, oran) okunur, temizlenir, çift kayıtlar atılır.
#
# 4c — Mevsimsel impütasyon:
#   Doluluk verisinin bulunmadığı günler (2020-2022 arası ve 2026 tahmini)
#   için aynı takvim ayının gerçek ortalaması kullanılır.
#   Türetilen özellikler:
#     dol_roll7  — 7 günlük ortalama (haftalık trend)
#     dol_delta  — günlük değişim (ivme sinyali)
#
# ⚠️  DİKKAT: df_raw.iloc[:, 2] tarih sütununun Excel'de 3. kolonda (C) olduğunu
#   varsayar. Excel yapın farklıysa bu indeksi güncelle.
#   Benzer şekilde 'Primer Doktor Kodu' sütun adı değişmişse prov_monthly
#   hesabı hata verir.
#
# ── 4a: Ameliyat verisi ──────────────────────────────────────────────────────
df_raw = pd.read_excel(DATA_PATH)
df_raw['date'] = pd.to_datetime(df_raw.iloc[:, 2], errors='coerce').dt.normalize()
df_raw = df_raw.dropna(subset=['date'])

daily_raw  = df_raw.groupby('date').size().reset_index(name='volume')
full_range = pd.date_range(daily_raw['date'].min(), daily_raw['date'].max(), freq='D')
daily = (daily_raw.set_index('date')
                  .reindex(full_range, fill_value=0)
                  .reset_index()
                  .rename(columns={'index': 'date'}))

df_raw['month_key'] = df_raw['date'].dt.to_period('M')
prov_monthly = (df_raw.groupby('month_key')['Primer Doktor Kodu']
                      .nunique().reset_index(name='num_prov'))

print(f'✔ Ameliyat: {len(df_raw):,} kayıt yüklendi')
print(f'  Tarih aralığı : {daily["date"].min().date()} → {daily["date"].max().date()}')
print(f'  Toplam gün    : {len(daily)}  (veri olan: {(daily.volume > 0).sum()})')
print(f'  Günlük hacim  : min={daily.volume.min()}  maks={daily.volume.max()}  ort={daily.volume.mean():.1f}')

# ── 4b: Doluluk verisi yükle ─────────────────────────────────────────────────
df_dol = pd.read_excel(DOLULUK_PATH)
df_dol.columns = ['date', 'doluluk']
df_dol['date']    = pd.to_datetime(df_dol['date']).dt.normalize()
df_dol['doluluk'] = pd.to_numeric(df_dol['doluluk'], errors='coerce')
df_dol = df_dol.dropna().drop_duplicates('date').set_index('date').sort_index()

print(f'\n✔ Doluluk: {len(df_dol):,} gün yüklendi')
print(f'  Tarih aralığı : {df_dol.index.min().date()} → {df_dol.index.max().date()}')
print(f'  Aralık        : {df_dol.doluluk.min():.2f} – {df_dol.doluluk.max():.2f}  (ort={df_dol.doluluk.mean():.3f})')

# ── 4c: Mevsimsel impütasyon ─────────────────────────────────────────────────
# Gerçek veriden aylık ortalama hesapla (ay bazında mevsimsel desen)
dol_monthly_avg = df_dol.copy()
dol_monthly_avg['month'] = dol_monthly_avg.index.month
monthly_avg = dol_monthly_avg.groupby('month')['doluluk'].mean()

# Tüm gün aralığı için doluluk serisi (ameliyat + tahmin dönemi)
all_dates  = pd.date_range(daily['date'].min(), '2026-03-31', freq='D')
dol_full   = pd.Series(index=all_dates, dtype=float, name='doluluk')
dol_full.update(df_dol['doluluk'])
# Eksik günleri aynı aydaki gerçek ortalamayla doldur
for dt in dol_full[dol_full.isna()].index:
    dol_full[dt] = monthly_avg.get(dt.month, monthly_avg.mean())

# Türetilmiş doluluk özellikleri
dol_roll7 = dol_full.rolling(7, min_periods=1, center=True).mean()
dol_delta = dol_full.diff().fillna(0)

# daily DataFrame'e ekle
daily['doluluk']   = daily['date'].map(dol_full).values
daily['dol_roll7'] = daily['date'].map(dol_roll7).values
daily['dol_delta'] = daily['date'].map(dol_delta).values

print(f'\n✔ Doluluk özellikleri daily DataFrame\'e eklendi')
print(f'  Eksik günler mevsimsel ortalamayla dolduruldu: {dol_full.isna().sum()} eksik kaldı')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Türk Tatil Günleri (2019–2026)                                ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   İki tatil kümesi oluşturur ve birleştirir:
#
#   TR_HOL    — Sabit tarihli resmi ulusal tatiller (her yıl aynı gün):
#               1 Ocak, 23 Nisan, 1 Mayıs, 19 Mayıs, 15 Temmuz,
#               30 Ağustos, 29 Ekim
#
#   RELIGIOUS — Hicri takvime göre kayan dini tatiller:
#               Ramazan Bayramı (3 gün) + Kurban Bayramı (4 gün, şu an 3)
#               Her yıl manuel girilmiştir — 2027 için güncellemeyi unutma.
#
#   ALL_HOL   — TR_HOL | RELIGIOUS — Cell 6 ve Cell 10'da kullanılır.
#
# Genişletme önerileri:
#   • Köprü günlerini ekle: bazı yıllarda hükümet kararıyla tatil edilen
#     'köprü' günleri vaka hacmini etkiliyor olabilir. Bunları TR_HOL'a
#     pd.Timestamp('YYYY-MM-DD') satırı ekleyerek dahil edebilirsin.
#   • 2027 tahminine uzatmak istersen RELIGIOUS sözlüğüne o yılın
#     Ramazan ve Kurban Bayramı tarihlerini ekle.
#
TR_HOL = set()
for y in range(2019, 2027):
    for m, d in [(1,1),(4,23),(5,1),(5,19),(7,15),(8,30),(10,29)]:
        TR_HOL.add(pd.Timestamp(f'{y}-{m:02d}-{d:02d}'))

RELIGIOUS = {
    # Ramazan Bayramı (Eid al-Fitr)
    pd.Timestamp('2020-05-24'), pd.Timestamp('2020-05-25'), pd.Timestamp('2020-05-26'),
    pd.Timestamp('2021-05-13'), pd.Timestamp('2021-05-14'), pd.Timestamp('2021-05-15'),
    pd.Timestamp('2022-05-02'), pd.Timestamp('2022-05-03'), pd.Timestamp('2022-05-04'),
    pd.Timestamp('2023-04-21'), pd.Timestamp('2023-04-22'), pd.Timestamp('2023-04-23'),
    pd.Timestamp('2024-04-10'), pd.Timestamp('2024-04-11'), pd.Timestamp('2024-04-12'),
    pd.Timestamp('2025-03-30'), pd.Timestamp('2025-03-31'), pd.Timestamp('2025-04-01'),
    pd.Timestamp('2026-03-20'), pd.Timestamp('2026-03-21'), pd.Timestamp('2026-03-22'),
    # Kurban Bayramı (Eid al-Adha)
    pd.Timestamp('2020-07-31'), pd.Timestamp('2020-08-01'), pd.Timestamp('2020-08-02'),
    pd.Timestamp('2021-07-20'), pd.Timestamp('2021-07-21'), pd.Timestamp('2021-07-22'),
    pd.Timestamp('2022-07-09'), pd.Timestamp('2022-07-10'), pd.Timestamp('2022-07-11'),
    pd.Timestamp('2023-06-28'), pd.Timestamp('2023-06-29'), pd.Timestamp('2023-06-30'),
    pd.Timestamp('2024-06-17'), pd.Timestamp('2024-06-18'), pd.Timestamp('2024-06-19'),
    pd.Timestamp('2025-06-06'), pd.Timestamp('2025-06-07'), pd.Timestamp('2025-06-08'),
}
ALL_HOL = TR_HOL | RELIGIOUS
print(f'✔ {len(ALL_HOL)} tatil günü kayıt edildi (2019–2026, ulusal + dini)')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Özellik Mühendisliği                                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   Her gün için modellerin göreceği tüm girdi değişkenlerini üretir.
#   Üretilen özellikler:
#
#   [Zaman]  day_of_year, week_of_year, year
#   [Döngüsel] sin/cos kodlama: dayofweek(7), month(12), quarter(4)
#              → Modelin 'Pazartesi ile Pazar arasındaki mesafeyi' doğru anlaması için.
#   [Tatil]  is_holiday, is_weekend — ikili bayraklar
#            near_hol_{-HOL_WINDOW..+HOL_WINDOW} — tatil öncesi/sonrası günler
#            HOL_WINDOW=3 ise 6 sütun üretilir (±1,±2,±3).
#   [Klinik] num_prov — o aydaki aktif cerrah sayısı
#   [Pandemi] is_covid — COVID_END tarihine kadar 1, sonrası 0
#   [Anomali] is_anomaly — IsolationForest ile işaretlenmiş aykırı günler
#   [Gecikmeli] lag_3/7/14/28/60/90 — geçmiş vaka sayıları
#               same_week_ly — geçen yıl aynı hafta (pandemi-farkındalıklı)
#   [Rolling] roll_7/14/28 — geçmişin hareketli ortalaması
#   [Doluluk] dol_ewma — DOL_EWMA_SPAN parametresiyle üstel ağırlıklı ortalama
#
# ⚠️  Parametre notları:
#   contamination=0.05 → toplam günlerin %5'i anomali işaretlenir (~115 gün).
#   [TEST ET] Pandemi baskısıyla bu çok yüksek olabilir; 0.03 ile dene.
#
#   lag_3 çok kısa vadeli ve gürültülü bir referanstır. Modelin performansına
#   katkısını SHAP veya feature importance ile kontrol et; gerekirse
#   FEATURE_COLS listesinden çıkar.
#
def engineer_features(df):
    df = df.copy()
    d  = pd.to_datetime(df['date'])

    # Zaman bileşenleri
    df['day_of_year']  = d.dt.dayofyear
    df['week_of_year'] = d.dt.isocalendar().week.astype(int)
    df['year']         = d.dt.year

    # Döngüsel kodlama — sin/cos ile periyodik mesafe korunur
    for feat, period in [('dayofweek', 7), ('month', 12), ('quarter', 4)]:
        v = getattr(d.dt, feat)
        df[f'sin_{feat}'] = np.sin(2 * np.pi * v / period)
        df[f'cos_{feat}'] = np.cos(2 * np.pi * v / period)

    # Tatil & hafta sonu ikili bayrakları
    df['is_holiday'] = d.isin(ALL_HOL).astype(int)
    df['is_weekend']  = (d.dt.dayofweek >= 5).astype(int)

    # HOL_WINDOW parametresiyle tatil yakınlık bayrakları
    # HOL_WINDOW=3 → near_hol_-3, near_hol_-2, near_hol_-1, near_hol_+1, near_hol_+2, near_hol_+3
    for off in range(-HOL_WINDOW, HOL_WINDOW + 1):
        if off == 0: continue
        shifted = {h + pd.Timedelta(days=off) for h in ALL_HOL}
        df[f'near_hol_{off:+d}'] = d.isin(shifted).astype(int)

    # Aylık doktor sayısı — ameliyat kapasitesinin proxy'si
    mk = d.dt.to_period('M')
    pm = prov_monthly.set_index('month_key')['num_prov']
    df['num_prov'] = mk.map(pm).ffill().bfill()

    # COVID_END parametresiyle pandemi bayrağı
    df['is_covid'] = ((d >= '2020-01-01') & (d < COVID_END)).astype(int)

    # Anomali bayrağı — IsolationForest
    # [TEST ET] contamination=0.05 → %5 anomali. 0.03 ile dene.
    iso = IsolationForest(contamination=0.05, random_state=SEED)
    df['is_anomaly'] = (iso.fit_predict(df[['volume']]) == -1).astype(int)

    # Gecikmeli özellikler — geçmişi doğrudan girdi olarak sunar
    # lag_3: 3 gün öncesi (kısa vadeli, gürültülü) — performans düşükse çıkar
    # lag_90: 3 ay öncesi (mevsimsel bağlam)
    for lag in [3, 7, 14, 28, 60, 90]:
        df[f'lag_{lag}'] = df['volume'].shift(lag).fillna(0)

    # Geçen yıl aynı hafta (pandemi-farkındalıklı)
    # Pandemi dönemi düşük değerlerine fallback yapmamak için 2. ve 3. yıla bakar
    lag_1yr = df['volume'].shift(364)
    lag_2yr = df['volume'].shift(364 * 2)
    lag_3yr = df['volume'].shift(364 * 3)
    df['same_week_ly'] = lag_1yr.fillna(lag_2yr).fillna(lag_3yr).fillna(0)
    covid_mask     = (df['date'] >= '2022-01-01') & (df['date'] < '2024-01-01')
    covid_prev_low = df['same_week_ly'] < 20
    df.loc[covid_mask & covid_prev_low, 'same_week_ly'] = lag_2yr[covid_mask & covid_prev_low].fillna(0)

    # Rolling ortalama — kısa vadeli trendleri yakalar
    for win in [7, 14, 28]:
        df[f'roll_{win}'] = (df['volume'].shift(1)
                                        .rolling(win, min_periods=1)
                                        .mean().fillna(0))

    # EWMA doluluk — DOL_EWMA_SPAN parametresiyle son günlere üstel ağırlık
    # [TEST ET] span=3 hızlı tepki, span=14 daha düzgün ama gecikmeli sinyal
    df['dol_ewma'] = (df['doluluk'].ewm(span=DOL_EWMA_SPAN, adjust=False).mean())

    return df

daily = engineer_features(daily)

# FEATURE_COLS: HOL_WINDOW'a göre dinamik tatil sütunları dahil edilir
hol_cols = [f'near_hol_{off:+d}'
            for off in range(-HOL_WINDOW, HOL_WINDOW + 1) if off != 0]

FEATURE_COLS = (
    ['volume',
     'day_of_year', 'week_of_year',
     'sin_dayofweek', 'cos_dayofweek',
     'sin_month',    'cos_month',
     'sin_quarter',  'cos_quarter',
     'is_holiday', 'is_weekend']
    + hol_cols
    + ['num_prov', 'is_covid', 'is_anomaly',
       'lag_3', 'lag_7', 'lag_14', 'lag_28', 'lag_60', 'lag_90',
       'same_week_ly',
       'roll_7', 'roll_14', 'roll_28',
       'doluluk', 'dol_roll7', 'dol_delta', 'dol_ewma']
)

print(f'✔ Özellik mühendisliği tamamlandı')
print(f'  Toplam özellik : {len(FEATURE_COLS)}')
print(f'  Tatil yakınlık : {len(hol_cols)} sütun (HOL_WINDOW={HOL_WINDOW})')
print(f'  Anomali sayısı : {daily["is_anomaly"].sum()} gün ({daily["is_anomaly"].mean()*100:.1f}%)')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Eğitim/Validasyon Bölünmesi & Ölçekleme                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#
# 1. Zaman serisi bölünmesi:
#    train_df → 2020–2024 (TRAIN_END öncesi)
#    val_df   → 2025 (TRAIN_END ile VAL_END arası)
#    Karıştırma yapılmaz — zaman serisi sırası korunur.
#
# 2. RobustScaler:
#    Her özelliği medyan ve IQR'a göre ölçekler: (x - median) / IQR
#    MinMaxScaler'dan farkı: pandemi dönemindeki sıfıra yakın veya aşırı
#    yüksek günler ölçeklemeyi bozmaz, aykırı değerlere dayanıklıdır.
#    Scaler SADECE eğitim verisine fit edilir (data leakage önlenir);
#    val ve all setleri transform ile dönüştürülür.
#
# 3. Dizi üretimi (make_sequences):
#    LOOKBACK=28, HORIZON=7 için:
#    X[i] → 28 günlük pencere (tüm özellikler)
#    y[i] → sonraki 7 günün vaka sayısı
#    Toplam dizi = len(veri) - LOOKBACK - HORIZON + 1
#
# 4. inv_vol fonksiyonu:
#    Ölçeklenmiş tahmin çıktısını orijinal vaka sayısına geri çevirir.
#    ⚠️  ÖNEMLI: volume sütununun FEATURE_COLS'ta ilk sırada (indeks 0)
#    kalması zorunlu — aksi halde inverse transform yanlış sütunu kullanır.
#
# [TEST ET] VAL_SPLIT = 0.15 → LSTM iç validasyonu için eğitim setinin
#    %15'i ayrılır. Eğitim verisi az görünüyorsa 0.10'a düşür;
#    overfitting izliyorsan 0.20'ye çık.
#
train_df = daily[daily['date'] <  TRAIN_END].copy()
val_df   = daily[(daily['date'] >= TRAIN_END) &
                 (daily['date'] <  VAL_END)].copy()

scaler   = RobustScaler()  # median & IQR bazlı — aykırı değerlere dayanıklı
train_sc = scaler.fit_transform(train_df[FEATURE_COLS].values)
val_sc   = scaler.transform(val_df[FEATURE_COLS].values)
all_sc   = scaler.transform(daily[FEATURE_COLS].values)

def inv_vol(sv):
    """Ölçeklenmiş hacim değerlerini orijinal ölçeğe geri çevir.
    volume FEATURE_COLS'un ilk elemanı (indeks 0) olmalı.
    """
    d = np.zeros((len(sv), len(FEATURE_COLS)))
    d[:, 0] = sv
    return scaler.inverse_transform(d)[:, 0]

def make_sequences(arr, lb=LOOKBACK, hz=HORIZON):
    """(LOOKBACK, n_features) → HORIZON boyutunda dizi çiftleri üretir."""
    X, y = [], []
    for i in range(len(arr) - lb - hz + 1):
        X.append(arr[i : i + lb])
        y.append(arr[i + lb : i + lb + hz, 0])
    return np.array(X), np.array(y)

X_tr, y_tr = make_sequences(train_sc)
X_va, y_va = make_sequences(np.concatenate([train_sc[-LOOKBACK:], val_sc]))

print(f'Eğitim : {len(train_df):,} gün  ({train_df["date"].min().date()} – {train_df["date"].max().date()})')
print(f'Val    : {len(val_df):,} gün   ({val_df["date"].min().date()} – {val_df["date"].max().date()})')
print(f'LSTM dizisi — Eğitim: {len(X_tr):,}  |  Val: {len(X_va):,}')
print(f'Dizi şekli: {X_tr.shape}  →  Hedef şekli: {y_tr.shape}')
print(f'Scaler   : RobustScaler  (center={scaler.center_[0]:.2f}  scale={scaler.scale_[0]:.2f})')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — MODEL 1: Encoder-Decoder seq2seq LSTM                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#
# Mimari (build_lstm fonksiyonu):
#   Encoder:
#     • Bidirectional LSTM — hem ileriye hem geriye okuyarak
#       kısa vadeli örüntüleri çift yönlü yakalar
#     • Standart LSTM — son gizli durum [h, c]'yi decoder'a aktarır
#
#   Decoder:
#     • RepeatVector: encoder çıktısını HORIZON kez tekrarlar
#     • Decoder LSTM-1 (LSTM_UNITS): encoder gizli durumunu initial state alır
#     • Decoder LSTM-2 (LSTM_UNITS//2=32): daralan huni yapısı, daha iyi genelleme
#       [TEST ET] LSTM_UNITS//4=16 ile daha dar şişe boynu deneyebilirsin
#     • TimeDistributed(Dense(1)): her adım için tek tahmin
#
# Kayıp fonksiyonu:
#   Huber(delta=HUBER_DELTA) — delta altındaki hatalar MSE gibi (pürüzsüz gradyan),
#   üstündekiler MAE gibi (aykırı değerlere dayanıklı). Pandemi dönemi
#   sıfır-vaka günleri için bu denge kritik.
#
# Callbacks:
#   EarlyStopping  → val_loss EARLY_PATIENCE epoch iyileşmezse durur,
#                    en iyi ağırlıkları geri yükler
#   ReduceLROnPlateau → REDUCE_PATIENCE epoch iyileşmezse LR'yi REDUCE_FACTOR
#                       ile çarpar (min: 1e-7)
#
# [TEST ET] REDUCE_FACTOR=0.3 daha agresif LR düşürümü sağlar;
#   kayıp eğrisinde platoya daha erken oturup oturmadığını izle.
#
def build_lstm(n_features):
    enc_in = Input(shape=(LOOKBACK, n_features), name='encoder_input')

    # Bidirectional LSTM — çift yönlü örüntü öğrenimi
    x      = Bidirectional(
                 LSTM(LSTM_UNITS, return_sequences=True,
                      dropout=DROPOUT,
                      recurrent_dropout=RECURRENT_DROPOUT),
                 name='bi_lstm'
             )(enc_in)
    _, h, c = LSTM(LSTM_UNITS, return_state=True,
                   dropout=DROPOUT,
                   recurrent_dropout=RECURRENT_DROPOUT,
                   name='encoder_lstm')(x)

    # Decoder — 2 katmanlı daralan yapı (encoder gizli durumu initial state)
    # 1. katman: LSTM_UNITS genişlik, encoder bağlamını alır
    # 2. katman: LSTM_UNITS//2 genişlik, bilgiyi sıkıştırarak çıkışa hazırlar
    dec  = RepeatVector(HORIZON, name='repeat')(h)
    dec  = LSTM(LSTM_UNITS, return_sequences=True,
                dropout=DROPOUT, recurrent_dropout=RECURRENT_DROPOUT,
                name='decoder_lstm_1')(dec, initial_state=[h, c])
    dec  = LSTM(LSTM_UNITS // 2, return_sequences=True,
                dropout=DROPOUT,
                name='decoder_lstm_2')(dec)
    out  = TimeDistributed(Dense(1), name='output')(dec)
    out  = tf.keras.layers.Reshape((HORIZON,))(out)

    model = Model(enc_in, out, name='seq2seq_bilstm_v5')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE,
            clipnorm=CLIP_NORM
        ),
        loss=tf.keras.losses.Huber(delta=HUBER_DELTA)
    )
    return model

lstm_model = build_lstm(len(FEATURE_COLS))
lstm_model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=EARLY_PATIENCE,
                  restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=REDUCE_FACTOR,
                      patience=REDUCE_PATIENCE, min_lr=1e-7, verbose=1),
]

print(f'\n⏳ LSTM eğitiliyor …')
print(f'   LR={LEARNING_RATE}  clipnorm={CLIP_NORM}  Huber_δ={HUBER_DELTA}')
print(f'   recurrent_dropout={RECURRENT_DROPOUT}  patience={EARLY_PATIENCE}')
print(f'   (GPU ile ~5-12 dk, CPU ile ~40-60 dk)')
history = lstm_model.fit(
    X_tr, y_tr,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=callbacks,
    verbose=1
)
print(f'✔ LSTM eğitimi tamamlandı ({len(history.history["loss"])} epoch)')

# Kayıp eğrisi — eğitim ve val kaybı birbirinden çok ayrışıyorsa
# DROPOUT artırımını veya VAL_SPLIT büyütmeyi düşün
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history['loss'],     color='#3B82F6', lw=1.5, label='Eğitim Kaybı')
ax.plot(history.history['val_loss'], color='#F59E0B', lw=1.5, ls='--', label='Val Kaybı')
ax.set_title(f'LSTM — Eğitim & Validasyon Kaybı (Huber δ={HUBER_DELTA})', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Kayıp')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — LSTM Tahmin Fonksiyonu (Direct Forecast)                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   direct_forecast_lstm fonksiyonu HORIZON=7 günlük bloklarla tahmin üretir.
#
# Neden 'direct' (doğrudan) yöntem:
#   Klasik multi-step tahmininde model kendi tahminini bir sonraki adımın
#   girdisi olarak kullanır → hatalar birikir (error accumulation).
#   Burada her blok için son LOOKBACK günün GERÇEK verisi kullanılır,
#   bu nedenle hata birikimi oluşmaz.
#
# Validasyon modu:
#   Her i adımında: eğitim verisi + val_sc[:i] (o ana kadar görülen gerçek val)
#   Gerçekçi bir walk-forward simülasyonu yapar.
#
# Tahmin modu:
#   Tüm bilinen veri (2020-2025 all_sc) kullanılır, son LOOKBACK günü pencere.
#   ⚠️  2025 yılı verisi eksik veya seyrekse 2026 tahmini bu pencereden
#   olumsuz etkilenir — veriyi kontrol et.
#
def direct_forecast_lstm(mode='val'):
    if mode == 'val':
        target_dates = val_df['date'].values
        n_target     = len(val_df)
    else:
        target_dates = pd.date_range(FORECAST_START, FORECAST_END, freq='D')
        n_target     = len(target_dates)

    results = []
    dates   = pd.DatetimeIndex(pd.to_datetime(list(target_dates)))

    for i in range(0, n_target, HORIZON):
        chunk = dates[i : i + HORIZON]

        if mode == 'val':
            # Gerçek eğitim + o ana kadarki gerçek val verisi
            real_past = np.vstack([train_sc, val_sc[:i]])
        else:
            # Tüm bilinen gerçek veri (2020-2025)
            real_past = all_sc

        seed  = real_past[-LOOKBACK:]
        x_in  = seed.reshape(1, LOOKBACK, len(FEATURE_COLS))
        ps    = lstm_model.predict(x_in, verbose=0)[0]
        pv    = np.clip(inv_vol(ps), 0, None)  # Negatif tahmin sıfıra çekilir

        for dt, p in zip(chunk, pv):
            results.append({'date': pd.Timestamp(dt), 'pred_lstm': round(float(p))})

    return pd.DataFrame(results)

fcast_dates = pd.date_range(FORECAST_START, FORECAST_END, freq='D')

print('⏳ LSTM validasyon tahmini …')
lstm_val  = direct_forecast_lstm(mode='val')
print('⏳ LSTM 2026 tahmini …')
lstm_test = direct_forecast_lstm(mode='test')
print(f'✔ LSTM: {len(lstm_val)} validasyon, {len(lstm_test)} tahmin')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — MODEL 2: Facebook Prophet                                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   Prophet trend + mevsimsellik + tatil + dışsal regressor bileşenleriyle
#   ayrıştırılmış bir zaman serisi modeli kurar.
#
# Bileşenler:
#   • changepoint_prior_scale  → trendin ne kadar esnek değiştiği
#   • seasonality_mode=multiplicative → oransal mevsimsel etki
#     (yaz aylarında vaka %30 düşüyorsa additive yanlış ölçekler)
#   • yearly_seasonality=PROPHET_YEARLY_N → Fourier bileşeni sayısı
#   • holidays → ALL_HOL ile Türk tatilleri, ±2 gün pencereli
#   • Regressorlar:
#     num_prov — aylık cerrah sayısı (kapasite proxy'si)
#     is_covid — pandemi bayrağı (sadece eğitim döneminde 1)
#     doluluk  — hastane doluluk oranı
#
# ⚠️  Önemli notlar:
#   • is_covid 2026 tahmin döneminde sıfır geçilir — doğru.
#   • doluluk 2026 için mevsimsel ortalamaya (Cell 4 impütasyonu) fallback
#     yapar. Gerçek 2026 doluluk verisi elde edilirse dol_full serisini
#     güncelle ve bu hücreyi yeniden çalıştır.
#
# [TEST ET] PROPHET_CP_SCALE — Bu modelin en kritik parametresi.
#   Bileşen grafiğinde trend düz/garip görünüyorsa:
#     Çok düz → 0.3 veya 0.5'e çık
#     Çok kıvrımlı / overfit → 0.05'e düşür
#
print('⏳ Prophet eğitiliyor …')

# Tatil DataFrame'i: her tatil günü için ±2 gün etkisi modellenir
hol_df = pd.DataFrame([
    {'holiday': 'tr_tatil', 'ds': h,
     'lower_window': -2, 'upper_window': 2}
    for h in ALL_HOL
])

def get_prov(dt):
    mk = pd.Period(pd.Timestamp(dt), 'M')
    pm = prov_monthly.set_index('month_key')['num_prov']
    return float(pm.get(mk, pm.iloc[-1]))

def get_doluluk(dt):
    """Tarih için doluluk oranı döndür (impüte edilmiş dol_full serisinden)."""
    ts = pd.Timestamp(dt).normalize()
    return float(dol_full.get(ts, dol_full.mean()))

prophet_model = Prophet(
    seasonality_mode=PROPHET_SEAS_MODE,        # multiplicative: oransal mevsimsel etki
    weekly_seasonality=True,
    yearly_seasonality=PROPHET_YEARLY_N,       # Fourier bileşeni sayısı
    daily_seasonality=False,
    holidays=hol_df,
    changepoint_prior_scale=PROPHET_CP_SCALE,  # [TEST ET] trend esnekliği
    seasonality_prior_scale=PROPHET_SEAS_PRIOR,
    holidays_prior_scale=PROPHET_HOL_PRIOR,
)
prophet_model.add_regressor('num_prov', standardize=True)
prophet_model.add_regressor('is_covid', standardize=False)
prophet_model.add_regressor('doluluk',  standardize=True)

train_prophet = train_df[['date','volume','is_covid']].rename(
    columns={'date':'ds','volume':'y'}).copy()
train_prophet['num_prov'] = train_prophet['ds'].apply(get_prov)
train_prophet['doluluk']  = train_prophet['ds'].apply(get_doluluk)
prophet_model.fit(train_prophet)

n_future = len(val_df) + len(fcast_dates)
future   = prophet_model.make_future_dataframe(periods=n_future, freq='D')
future['num_prov'] = future['ds'].apply(get_prov)
future['is_covid'] = ((future['ds'] >= '2020-01-01') &
                      (future['ds'] <  '2022-06-01')).astype(int)
future['doluluk']  = future['ds'].apply(get_doluluk)
fc_prophet = prophet_model.predict(future)

prop_val  = (fc_prophet[fc_prophet['ds'].isin(val_df['date'])]
             [['ds','yhat']]
             .rename(columns={'ds':'date','yhat':'pred_prophet'}))
prop_val['pred_prophet'] = np.clip(prop_val['pred_prophet'], 0, None).round()

prop_test = (fc_prophet[fc_prophet['ds'].isin(fcast_dates)]
             [['ds','yhat']]
             .rename(columns={'ds':'date','yhat':'pred_prophet'}))
prop_test['pred_prophet'] = np.clip(prop_test['pred_prophet'], 0, None).round()

print('✔ Prophet tamamlandı')
print('ℹ️  Bileşen grafiği: trend düz/garip görünüyorsa PROPHET_CP_SCALE değerini ayarla')

# Bileşen grafiği — trend, haftalık mevsimsellik, yıllık mevsimsellik, tatil
# etkilerini ayrı ayrı gösterir. Beklenen görünüm:
#   • Trend: düzgün yükselen, pandemi döneminde belirgin çukur
#   • Weekly: Pzt-Cum yüksek, hafta sonu düşük
#   • Yearly: yaz aylarında düşüş, Ocak-Mart yüksek
fig = prophet_model.plot_components(fc_prophet)
fig.suptitle('Prophet — Trend & Mevsimsellik Bileşenleri (2020–2026)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — MODEL 3: LightGBM                                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   HORIZON=7 adımın her biri için bağımsız bir LGBMRegressor eğitir.
#   (7 model × LGB_N_ESTIMATORS ağaç)
#
# Neden her adım için ayrı model:
#   MultiOutputRegressor yerine bu yaklaşım, her adımın farklı örüntüleri
#   (Pzt. 1 gün ileride vs Paz. 7 gün ileride) bağımsız öğrenmesine izin verir.
#
# Girdi yapısı:
#   make_seq_flat: (LOOKBACK × n_features) boyutunda düzleştirilmiş vektör
#   Her pencere tüm özellik geçmişini düz bir dizi olarak alır.
#
# ⚠️  Early stopping henüz eklenmedi — aşağıdaki kodu ekleyerek
#   overfitting'i önleyebilir ve LGB_N_ESTIMATORS'ı güvenle artırabilirsin:
#
#   # Cell 7'nin sonuna ekle:
#   X_va_f, y_va_f = make_seq_flat(val_sc)
#
#   # m.fit(...) satırını şununla değiştir:
#   m.fit(X_tr_f, y_tr_f[:, step],
#         eval_set=[(X_va_f, y_va_f[:, step])],
#         callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
#
#   Early stopping eklenirse LGB_N_ESTIMATORS 500-1000'e güvenle çıkabilir.
#
print('⏳ LightGBM eğitiliyor …')

def make_seq_flat(arr, lb=LOOKBACK, hz=HORIZON):
    """LSTM dizilerini LightGBM için (LOOKBACK*n_features,) düz vektöre çevirir."""
    X, y = [], []
    for i in range(len(arr) - lb - hz + 1):
        X.append(arr[i : i + lb].flatten())
        y.append(arr[i + lb : i + lb + hz, 0])
    return np.array(X), np.array(y)

X_tr_f, y_tr_f = make_seq_flat(train_sc)

# Her HORIZON adımı için ayrı model — bağımsız öğrenme
lgb_models = []
for step in range(HORIZON):
    m = lgb.LGBMRegressor(
        n_estimators     = LGB_N_ESTIMATORS,  # [TEST ET] early stopping ile 500+ dene
        max_depth        = LGB_MAX_DEPTH,     # [TEST ET] 4 daha basit, 8 overfitting riski
        learning_rate    = LGB_LR,            # N_ESTIMATORS ile birlikte ayarla
        subsample        = LGB_SUBSAMPLE,
        subsample_freq   = 1,
        colsample_bytree = LGB_COLSAMPLE,
        min_child_samples= LGB_MIN_LEAF,      # [TEST ET] 3 az örnek durumlar için
        random_state     = SEED,
        n_jobs           = -1,
        verbose          = -1
    )
    m.fit(X_tr_f, y_tr_f[:, step])
    lgb_models.append(m)

print(f'✔ LightGBM tamamlandı ({HORIZON} model × {LGB_N_ESTIMATORS} ağaç)')
print(f'   depth={LGB_MAX_DEPTH}  lr={LGB_LR}  subsample={LGB_SUBSAMPLE}  col={LGB_COLSAMPLE}')

def direct_forecast_lgb(mode='val'):
    """LSTM ile aynı direct forecast mantığı — hata birikimi yok."""
    if mode == 'val':
        target_dates = val_df['date'].values
        n_target     = len(val_df)
    else:
        target_dates = fcast_dates
        n_target     = len(fcast_dates)

    results = []
    dates   = pd.DatetimeIndex(pd.to_datetime(list(target_dates)))

    for i in range(0, n_target, HORIZON):
        chunk = dates[i : i + HORIZON]
        if mode == 'val':
            real_past = np.vstack([train_sc, val_sc[:i]])
        else:
            real_past = all_sc
        seed = real_past[-LOOKBACK:]
        flat = seed.flatten().reshape(1, -1)
        ps   = np.array([m.predict(flat)[0] for m in lgb_models])
        pv   = np.clip(inv_vol(ps), 0, None)
        for dt, p in zip(chunk, pv):
            results.append({'date': pd.Timestamp(dt), 'pred_lgb': round(float(p))})

    return pd.DataFrame(results)

gbm_val  = direct_forecast_lgb(mode='val')
gbm_test = direct_forecast_lgb(mode='test')
# Sütun adı uyumluluğu için alias — Cell 12 pred_gbm sütununu okur
gbm_val['pred_gbm']  = gbm_val['pred_lgb']
gbm_test['pred_gbm'] = gbm_test['pred_lgb']
print(f'✔ LightGBM: {len(gbm_val)} validasyon, {len(gbm_test)} tahmin')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Metrikler, Karşılaştırma & Ağırlıklı Ensemble               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#
# 1. compute_metrics: R², RMSE, MAE hesaplar.
#    Not: yalnızca volume>0 olan günler değerlendirmeye alınır;
#    tatil/hafta sonu sıfırları metrikleri bozmaz.
#
# 2. Metrik tablosu: üç modeli referans paper sonucuyla karşılaştırır.
#    Durum eşikleri: R²≥0.7 → İyi, ≥0.4 → Orta, <0.4 → Düşük
#
# 3. Ağırlıklı Ensemble (v5 yeniliği):
#    Her modelin val R²'si ağırlık olarak kullanılır:
#      w_i = max(R²_i, 0) / Σ max(R²_j, 0)
#    Negatif R² → 0 ağırlık (modeli ensemble'dan çıkarır)
#    Ensemble tahmini = Σ w_i × pred_i
#
# [TEST ET] Alternatif ağırlıklandırma — 1/RMSE:
#    Büyük sapma yapan modeli daha agresif cezalandırır.
#    Aşağıdaki w_raw satırını değiştirerek dene:
#
#    rmse_lstm = compute_metrics(val_comp['volume'], val_comp['pred_lstm'])[1]
#    rmse_prop = compute_metrics(val_comp['volume'], val_comp['pred_prophet'])[1]
#    rmse_gbm  = compute_metrics(val_comp['volume'], val_comp['pred_gbm'])[1]
#    w_raw = np.array([1/rmse_lstm, 1/rmse_prop, 1/rmse_gbm])
#
def compute_metrics(y_true, y_pred):
    yt = np.array(y_true); yp = np.array(y_pred)
    m  = yt > 0  # Sıfır vaka günlerini dışla
    return (r2_score(yt[m], yp[m]),
            np.sqrt(mean_squared_error(yt[m], yp[m])),
            mean_absolute_error(yt[m], yp[m]))

def build_comparison(actual_df, *pred_dfs):
    merged = actual_df[['date','volume']].copy()
    for pdf in pred_dfs:
        merged = merged.merge(pdf, on='date', how='left')
    return merged.fillna(0)

val_comp = build_comparison(val_df, lstm_val, prop_val, gbm_val)

# Gerçek 2026 verisi (dosyada varsa kullan — has_actual bayrağı Cell 13'te aktif olur)
actual_2026 = daily[daily['date'] >= pd.Timestamp(FORECAST_START)][['date','volume']]

test_comp = pd.DataFrame({'date': fcast_dates})
for pdf in [lstm_test, prop_test, gbm_test]:
    test_comp = test_comp.merge(pdf, on='date', how='left')
test_comp = test_comp.merge(actual_2026, on='date', how='left')
test_comp = test_comp.fillna(0)
test_comp['is_weekend'] = pd.DatetimeIndex(test_comp['date']).dayofweek >= 5
test_comp['is_holiday'] = pd.DatetimeIndex(test_comp['date']).isin(ALL_HOL)

# Metrik tablosu
print('='*62)
print(f"  Validasyon Performansı — 2025")
print(f"  {'Model':<16} {'R²':>6} {'RMSE':>7} {'MAE':>7}  {'Durum':<12}")
print('  ' + '─'*56)
for col, name in [('pred_lstm','LSTM'),('pred_prophet','Prophet'),('pred_gbm','LightGBM')]:
    r2, rm, ma = compute_metrics(val_comp['volume'], val_comp[col])
    durum = '✅ İyi' if r2 >= 0.7 else ('⚠️ Orta' if r2 >= 0.4 else '❌ Düşük')
    print(f"  {name:<16} {r2:>6.3f} {rm:>7.2f} {ma:>7.2f}  {durum}")
print(f"  {'Paper LSTM':<16} {'0.855':>6} {'2.017':>7} {'1.104':>7}  (referans)")
print('='*62)

# Ağırlıklı Ensemble — Val R²'ye göre
print()
print('='*62)
print('  Ağırlıklı Ensemble (val R²)')

R2_lstm    = compute_metrics(val_comp['volume'], val_comp['pred_lstm'])[0]
R2_prophet = compute_metrics(val_comp['volume'], val_comp['pred_prophet'])[0]
R2_gbm     = compute_metrics(val_comp['volume'], val_comp['pred_gbm'])[0]

# Negatif R²'yi 0'a çek — o model ensemble'a katkısız kabul edilir
w_raw = np.array([max(R2_lstm, 0), max(R2_prophet, 0), max(R2_gbm, 0)])
if w_raw.sum() == 0:
    w_raw = np.ones(3)  # Tüm modeller başarısızsa eşit ağırlık
weights   = w_raw / w_raw.sum()
w_lstm, w_prophet, w_gbm = weights

print(f'  Ağırlıklar: LSTM={w_lstm:.3f}  Prophet={w_prophet:.3f}  LightGBM={w_gbm:.3f}')

val_comp['pred_ensemble'] = (
    w_lstm    * val_comp['pred_lstm'] +
    w_prophet * val_comp['pred_prophet'] +
    w_gbm     * val_comp['pred_gbm']
).round()
test_comp['pred_ensemble'] = (
    w_lstm    * test_comp['pred_lstm'] +
    w_prophet * test_comp['pred_prophet'] +
    w_gbm     * test_comp['pred_gbm']
).round()

r2e, rme, mae_e = compute_metrics(val_comp['volume'], val_comp['pred_ensemble'])
durum_e = '✅ İyi' if r2e >= 0.7 else ('⚠️ Orta' if r2e >= 0.4 else '❌ Düşük')
print(f'  Ensemble         {r2e:>6.3f} {rme:>7.2f} {mae_e:>7.2f}  {durum_e}')
print('='*62)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 13 — Ana Görselleştirme (6 Satır × 3 Sütun GridSpec)             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   Tüm modellerin performansını ve 2026 tahminlerini tek bir büyük figürde
#   gösterir. GridSpec ile 6 satır × 3 sütun ızgara düzeni kurulur.
#
# Satır düzeni:
#   Satır 0 (tam genişlik) — 2025 Validasyon: Gerçek vs tüm modeller + ensemble
#   Satır 1 (tam genişlik) — 2026 Tahmin: hafta sonu/tatil gri bantlarıyla
#   Satır 2 (3 sütun)     — Scatter: her model için tahmin vs gerçek (validasyon)
#   Satır 3 (tam genişlik) — Haftalık toplam karşılaştırma
#   Satır 4 (tam genişlik) — Aylık toplam karşılaştırma
#   Satır 5 (tam genişlik) — Hata dağılımı (residuals)
#
# has_actual bayrağı:
#   Gerçek 2026 verisi (test_comp['volume'].sum()>0) varsa
#   metrikler de 2026 grafiğine eklenir. Veri geldikçe otomatik aktif olur.
#
COLS = {'LSTM':'#3B82F6', 'Prophet':'#F59E0B', 'GBM':'#10B981', 'Ensemble':'#8B5CF6', 'Actual':'#1E293B'}
has_actual = test_comp['volume'].sum() > 0

fig = plt.figure(figsize=(18, 34), facecolor='#F8FAFC')
gs  = gridspec.GridSpec(6, 3, figure=fig, hspace=0.52, wspace=0.32,
                        top=0.95, bottom=0.03, left=0.06, right=0.97)

# ── Satır 0: Validasyon 2025 ──────────────────────────────────────────────────
# Tüm modeller ve ensemble gerçek veriyle kıyaslanır.
# Ensemble çizgisi solid (-) ile diğerlerinden ayırt edilir.
ax0 = fig.add_subplot(gs[0, :]); ax0.set_facecolor('white')
ax0.plot(val_df['date'], val_df['volume'],
         color=COLS['Actual'], lw=2, label='Gerçek', zorder=5)
for col, name, ls in [('pred_lstm','LSTM','--'),
                       ('pred_prophet','Prophet',':'),
                       ('pred_gbm','LightGBM','-.'),
                       ('pred_ensemble','Ensemble','-')]:
    r2, rm, ma = compute_metrics(val_comp['volume'], val_comp[col])
    ax0.plot(val_df['date'], val_comp[col], color=COLS.get(name, COLS['GBM']), lw=1.7, ls=ls.strip(),
             label=f'{name:<12} R²={r2:.3f}  RMSE={rm:.1f}  MAE={ma:.1f}')
ax0.set_title('Validasyon — 2025: Tüm Modeller + Ensemble vs Gerçek Günlük Vaka',
              fontsize=12, fontweight='bold')
ax0.set_ylabel('Günlük Vaka')
ax0.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax0.xaxis.set_major_locator(mdates.MonthLocator())
plt.setp(ax0.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax0.legend(fontsize=9, ncol=2, loc='upper left', prop={'family':'monospace'})
ax0.grid(alpha=0.2, lw=0.5)

# ── Satır 1: Ocak-Mart 2026 Tahminleri ───────────────────────────────────────
# Gri bantlar: hafta sonu ve tatil günlerini vurgular.
# Gerçek 2026 verisi varsa (has_actual=True) bar grafik olarak gösterilir.
ax1 = fig.add_subplot(gs[1, :]); ax1.set_facecolor('white')
for _, r in test_comp[test_comp['is_weekend'] | test_comp['is_holiday']].iterrows():
    ax1.axvspan(r['date'] - pd.Timedelta(hours=12),
                r['date'] + pd.Timedelta(hours=12),
                alpha=0.07, color='gray')
if has_actual:
    ax1.bar(test_comp['date'], test_comp['volume'],
            color='#CBD5E1', width=0.85, label='Gerçek', zorder=2)
for col, name, ls in [('pred_lstm','LSTM','--'),
                       ('pred_prophet','Prophet',':'),
                       ('pred_gbm','LightGBM','-.'),
                       ('pred_ensemble','Ensemble','-')]:
    lbl = name
    if has_actual:
        r2, rm, ma = compute_metrics(test_comp['volume'], test_comp[col])
        lbl = f'{name}  R²={r2:.3f}  MAE={ma:.1f}'
    ax1.plot(test_comp['date'], test_comp[col], color=COLS.get(name, COLS['GBM']),
             lw=2.2, ls=ls.strip(), label=lbl, zorder=5)
ax1.set_title('Tahmin — Ocak–Mart 2026 (gri = hafta sonu/tatil)',
              fontsize=12, fontweight='bold')
ax1.set_ylabel('Günlük Vaka')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax1.legend(fontsize=9, ncol=4, prop={'family':'monospace'})
ax1.grid(alpha=0.2, lw=0.5)

# ── Satır 2: Scatter grafikleri (validasyon) ──────────────────────────────────
# İdeal tahmin: tüm noktalar y=x çizgisinin üzerinde.
# Sistematik sapma (noktalar hep bir tarafta) → modelin o bölgede bias'ı var.
for ci, (col, name) in enumerate([('pred_lstm','LSTM'),
                                    ('pred_prophet','Prophet'),
                                    ('pred_gbm','LightGBM')]):
    ax = fig.add_subplot(gs[2, ci]); ax.set_facecolor('white')
    mask = val_comp['volume'] > 0
    ax.scatter(val_comp.loc[mask,'volume'], val_comp.loc[mask,col],
               alpha=0.4, s=8, color=COLS.get(name, '#888'))
    mn = min(val_comp.loc[mask,'volume'].min(), val_comp.loc[mask,col].min())
    mx = max(val_comp.loc[mask,'volume'].max(), val_comp.loc[mask,col].max())
    ax.plot([mn,mx],[mn,mx], 'k--', lw=0.8, alpha=0.5)
    r2, rm, ma = compute_metrics(val_comp['volume'], val_comp[col])
    ax.set_title(f'{name}  R²={r2:.3f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Gerçek'); ax.set_ylabel('Tahmin')
    ax.grid(alpha=0.2, lw=0.5)

plt.suptitle('KUH Ameliyat Hacmi Tahmin Modeli — v5 Sonuçları', fontsize=14, fontweight='bold', y=0.97)
plt.savefig('kuh_forecast_v5.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Grafik kaydedildi: kuh_forecast_v5.png')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 14 — Modelleri Kaydet & İndir                                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ne yapar:
#   Eğitilmiş modelleri ve tahmin sonuçlarını kaydeder, Colab'dan indirir.
#
# Kaydedilen dosyalar:
#   kuh_lstm_v5_tuned.keras      — TensorFlow/Keras LSTM modeli
#                                   (tf.keras.models.load_model ile yükle)
#   kuh_lgb_scaler_v5.pkl        — LightGBM modelleri + RobustScaler +
#                                   FEATURE_COLS + ensemble ağırlıkları
#   kuh_tahmin_oca_mar_2026.csv  — 2026 Oca-Mar tahminleri (tüm modeller + ensemble)
#   kuh_validasyon_2025.csv      — 2025 validasyon sonuçları
#   kuh_forecast_v5.png          — ana görselleştirme grafiği
#
# ⚠️  Prophet modeli kaydedilmiyor. Kaydetmek için şu satırları ekle:
#   from prophet.serialize import model_to_json
#   with open('kuh_prophet_v5.json', 'w') as f:
#       f.write(model_to_json(prophet_model))
#   files.download('kuh_prophet_v5.json')
#
#   Yüklemek için:
#   from prophet.serialize import model_from_json
#   with open('kuh_prophet_v5.json', 'r') as f:
#       prophet_model = model_from_json(f.read())
#   Bu sayede 2027 tahmininde modeli yeniden eğitmeden kullanabilirsin.
#
from google.colab import files
import pickle

lstm_model.save('kuh_lstm_v5_tuned.keras')

with open('kuh_lgb_scaler_v5.pkl', 'wb') as f:
    pickle.dump({'lgb_models': lgb_models, 'scaler': scaler,
                 'feature_cols': FEATURE_COLS,
                 'weights': {'lstm': w_lstm, 'prophet': w_prophet, 'lgb': w_gbm}}, f)

test_comp.to_csv('kuh_tahmin_oca_mar_2026.csv', index=False)
val_comp.to_csv('kuh_validasyon_2025.csv', index=False)

for fname in ['kuh_forecast_v5.png',
              'kuh_tahmin_oca_mar_2026.csv',
              'kuh_validasyon_2025.csv',
              'kuh_lstm_v5_tuned.keras']:
    try:
        files.download(fname)
    except Exception as e:
        print(f'  ⚠️ {fname}: {e}')

print('\n✅ Tüm dosyalar indirildi!')
print('  📊 kuh_forecast_v5.png                — ana grafik (ensemble dahil)')
print('  📋 kuh_tahmin_oca_mar_2026.csv        — Oca-Mar 2026 tahminleri + ensemble')
print('  📋 kuh_validasyon_2025.csv            — 2025 validasyon sonuçları')
print('  🤖 kuh_lstm_v5_tuned.keras            — eğitilmiş LSTM modeli (v5)')
print('  📦 kuh_lgb_scaler_v5.pkl              — LightGBM modelleri + ensemble ağırlıkları')
print()
print('  ℹ️  Prophet modeli kaydedilmedi. Yukarıdaki yorumlara bak.')
